# Planner Agent

The **Planner** is the entry node of the LangGraph workflow. It:

1. Defines `ResearchState` — the shared TypedDict that flows through all agents
2. **Normalizes** the ticker symbol (uppercase, trimmed, exchange suffix preserved)
3. Acts as the fan-out coordinator — after it returns, LangGraph fires **News** and **Financials** in parallel

No LLM calls — pure coordination.

## Normalization rules

| Input | Normalized | Market |
|---|---|---|
| `" aapl "` | `AAPL` | US |
| `"reliance.ns"` | `RELIANCE.NS` | India NSE |
| `"tcs.bo"` | `TCS.BO` | India BSE |
| `"INFY.NS "` | `INFY.NS` | India NSE |

```
START → planner → ┌─ news ─────────── sentiment ─┐
                   │                               ├─ analyst → END
                   └─ financials ─────────────────┘
```

In [ ]:
from typing import TypedDict, Optional
from dotenv import load_dotenv

load_dotenv()


class ResearchState(TypedDict):
    """Shared state flowing through the entire LangGraph workflow."""

    ticker:          str            # e.g. "AAPL" or "RELIANCE.NS"
    news_articles:   Optional[list] # raw Tavily results
    news_summary:    Optional[str]  # 3-bullet GPT summary
    financials:      Optional[dict] # Yahoo Finance metrics
    sentiment_score: Optional[float]# 0.0 (bearish) → 1.0 (bullish)
    report:          Optional[str]  # final analyst report


def _normalize_ticker(raw: str) -> str:
    """
    Normalize a ticker symbol:
    - Strip whitespace
    - Uppercase everything
    - Preserve exchange suffix (.NS / .BO) for Indian tickers

    Examples:
        ' aapl '       → 'AAPL'
        'reliance.ns'  → 'RELIANCE.NS'
        'TCS.bo '      → 'TCS.BO'
    """
    return raw.strip().upper()


def planner_node(state: ResearchState) -> dict:
    """
    Entry node: normalize the ticker, initialize optional state fields to None.
    LangGraph fans out to news_node + financials_node after this returns.
    """
    ticker = _normalize_ticker(state["ticker"])

    # Detect market for logging
    if ticker.endswith(".NS"):
        market = "India NSE"
    elif ticker.endswith(".BO"):
        market = "India BSE"
    else:
        market = "US"

    print(f"[Planner] Starting research pipeline — {ticker} ({market})")

    return {
        "ticker":          ticker,
        "news_articles":   None,
        "news_summary":    None,
        "financials":      None,
        "sentiment_score": None,
        "report":          None,
    }


In [ ]:
if __name__ == "__main__":
    # --- Demo: normalization tests ---
    test_cases = [" aapl ", "reliance.ns", "TCS.bo ", "INFY.NS", "nvda"]

    for raw in test_cases:
        result = planner_node({"ticker": raw})
        print(f"  '{raw}'  →  '{result['ticker']}'")
